In [ ]:
# Delta Lake tables 
Use this notebook to explore Delta Lake functionality 

In [1]:
from pyspark.sql.types import StructType, IntegerType, StringType, DoubleType

# define the schema
schema = StructType() \
.add("ProductID", IntegerType(), True) \
.add("ProductName", StringType(), True) \
.add("Category", StringType(), True) \
.add("ListPrice", DoubleType(), True)

df = spark.read.format("csv").option("header","true").schema(schema).load("Files/products/products.csv")
# df now is a Spark DataFrame containing CSV data from "Files/products/products.csv".
display(df)

StatementMeta(, fc673c62-5c91-4cd7-9dd9-5017366b4505, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c348951d-fcba-4fae-90d2-db0d2c239d6c)

In [3]:
# To create a managed Delta table:
df.write.format("delta").saveAsTable("managed_products")


StatementMeta(, fc673c62-5c91-4cd7-9dd9-5017366b4505, 5, Finished, Available, Finished, False)

In [7]:
# To create an External Delta table:

df.write.format("delta").saveAsTable("external_products", path="abfss://c3e42ed9-8b66-4a96-8125-90bcfd9d0523@onelake.dfs.fabric.microsoft.com/0ccb703c-bc56-410d-86b1-da66dc884ae2/Files/external_products")


StatementMeta(, fc673c62-5c91-4cd7-9dd9-5017366b4505, 9, Finished, Available, Finished, False)

In [12]:
%%sql
DESCRIBE FORMATTED external_products;

StatementMeta(, fc673c62-5c91-4cd7-9dd9-5017366b4505, 14, Finished, Available, Finished, False)

<Spark SQL result set with 12 rows and 3 fields>

In [11]:
%%sql
DESCRIBE FORMATTED managed_products;

StatementMeta(, fc673c62-5c91-4cd7-9dd9-5017366b4505, 13, Finished, Available, Finished, False)

<Spark SQL result set with 12 rows and 3 fields>

In [13]:
%%sql
DROP TABLE managed_products;
DROP TABLE external_products;

StatementMeta(, fc673c62-5c91-4cd7-9dd9-5017366b4505, 16, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

In [14]:
%%sql
CREATE TABLE products
USING DELTA
LOCATION 'Files/external_products';

StatementMeta(, fc673c62-5c91-4cd7-9dd9-5017366b4505, 17, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [15]:
%%sql
SELECT * FROM products;

StatementMeta(, fc673c62-5c91-4cd7-9dd9-5017366b4505, 18, Finished, Available, Finished, False)

<Spark SQL result set with 295 rows and 4 fields>

In [16]:
%%sql
UPDATE products
SET ListPrice = ListPrice * 0.9
WHERE Category = 'Mountain Bikes';

StatementMeta(, fc673c62-5c91-4cd7-9dd9-5017366b4505, 19, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [17]:
%%sql
DESCRIBE HISTORY products;

StatementMeta(, fc673c62-5c91-4cd7-9dd9-5017366b4505, 20, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 15 fields>

In [18]:
delta_table_path = 'Files/external_products'
# Get the current data
current_data = spark.read.format("delta").load(delta_table_path)
display(current_data)

# Get the version 0 data
original_data = spark.read.format("delta").option("versionAsOf", 0).load(delta_table_path)
display(original_data)

StatementMeta(, fc673c62-5c91-4cd7-9dd9-5017366b4505, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 410ab1c1-3e79-465a-9eee-f08602858cfc)

SynapseWidget(Synapse.DataFrame, 1b58fce9-deae-475c-9bbd-0c8e0c3d53c7)

In [19]:
%%sql
-- Create a temporary view
CREATE OR REPLACE TEMPORARY VIEW products_view
AS
    SELECT Category, COUNT(*) AS NumProducts, MIN(ListPrice) AS MinPrice, MAX(ListPrice) AS MaxPrice, AVG(ListPrice) AS AvgPrice
    FROM products
    GROUP BY Category;

SELECT *
FROM products_view
ORDER BY Category;    

StatementMeta(, fc673c62-5c91-4cd7-9dd9-5017366b4505, 23, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 37 rows and 5 fields>

In [20]:
%%sql
SELECT Category, NumProducts
FROM products_view
ORDER BY NumProducts DESC
LIMIT 10;

StatementMeta(, fc673c62-5c91-4cd7-9dd9-5017366b4505, 24, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 2 fields>

In [21]:
from pyspark.sql.functions import col, desc

df_products = spark.sql("SELECT Category, MinPrice, MaxPrice, AvgPrice FROM products_view").orderBy(col("AvgPrice").desc())
display(df_products.limit(6))

StatementMeta(, fc673c62-5c91-4cd7-9dd9-5017366b4505, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8a96f476-b752-4129-b41c-ae00108b3c10)

#### Use Delta tables for streaming data:

In [22]:
from notebookutils import mssparkutils
from pyspark.sql.types import *
from pyspark.sql.functions import *

# Create a folder
inputPath = 'Files/data/'
mssparkutils.fs.mkdirs(inputPath)

# Create a stream that reads data from the folder, using a JSON schema
jsonSchema = StructType([
StructField("device", StringType(), False),
StructField("status", StringType(), False)
])
iotstream = spark.readStream.schema(jsonSchema).option("maxFilesPerTrigger", 1).json(inputPath)

# Write some event data to the folder
device_data = '''{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"ok"}
{"device":"Dev2","status":"error"}
{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"error"}
{"device":"Dev2","status":"ok"}
{"device":"Dev2","status":"error"}
{"device":"Dev1","status":"ok"}'''

mssparkutils.fs.put(inputPath + "data.txt", device_data, True)

print("Source stream created...")

StatementMeta(, fc673c62-5c91-4cd7-9dd9-5017366b4505, 26, Finished, Available, Finished, False)

Source stream created...


In [23]:
# Write the stream to a delta table
delta_stream_table_path = 'Tables/iotdevicedata'
checkpointpath = 'Files/delta/checkpoint'
deltastream = iotstream.writeStream.format("delta").option("checkpointLocation", checkpointpath).start(delta_stream_table_path)
print("Streaming to delta sink...")

StatementMeta(, fc673c62-5c91-4cd7-9dd9-5017366b4505, 27, Finished, Available, Finished, False)

Streaming to delta sink...


In [24]:
%%sql
SELECT * FROM IotDeviceData;

StatementMeta(, fc673c62-5c91-4cd7-9dd9-5017366b4505, 28, Finished, Available, Finished, False)

<Spark SQL result set with 9 rows and 2 fields>

In [25]:
# Add more data to the source stream
more_data = '''{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"error"}
{"device":"Dev2","status":"error"}
{"device":"Dev1","status":"ok"}'''

mssparkutils.fs.put(inputPath + "more-data.txt", more_data, True)

StatementMeta(, fc673c62-5c91-4cd7-9dd9-5017366b4505, 29, Finished, Available, Finished, False)

True

In [26]:
%%sql
SELECT * FROM IotDeviceData;

StatementMeta(, fc673c62-5c91-4cd7-9dd9-5017366b4505, 30, Finished, Available, Finished, False)

<Spark SQL result set with 16 rows and 2 fields>

In [27]:
deltastream.stop()


StatementMeta(, fc673c62-5c91-4cd7-9dd9-5017366b4505, 31, Finished, Available, Finished, True)